<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/14B_NeuroFHIR_Review_WISH_Final_Stimulus_and_Counterbalance_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-Review — Notebook 14B
## Final Stimulus + Counterbalance Validation for WISH 2026

Run this **after the executed Notebook 14** and **before recruiting participants**.

This notebook makes the final corrections:

1. Participant previews use **MRI + model-predicted contour only**, never the expert reference contour.
2. Each hidden sequence has:
   - Evidence-First: **3 stable + 3 progression**
   - AI-First: **3 stable + 3 progression**
3. Counterbalancing uses coordinator-issued sequential pseudonymous IDs:
   - `P001 -> A`
   - `P002 -> B`
   - `P003 -> A`
   - etc.
4. The researcher key remains separate from the participant package.
5. Final leakage and pilot-readiness checks run automatically.

**Boundary:** longitudinal priors and some workflow signals are standardized/synthetic research stimuli. They are not true repeated scans of the public-image donors. Path B remains a workflow study, not diagnostic validation.


In [1]:
# Cell 1 — Load Notebook 14 outputs

from __future__ import annotations
import csv, json, re, shutil
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT=Path("/content/drive/MyDrive/neurofhir-qc")
WISH_ROOT=PROJECT_ROOT/"wish_extension"
NB14_ROOT=WISH_ROOT/"notebook_14_distinct_case_study"

NB14_AUDIT=NB14_ROOT/"evaluation/notebook_14_distinct_case_protocol_hardening_audit.json"
NB14_RESEARCH=NB14_ROOT/"researcher_only/researcher_scenario_key.json"
NB14_PARTICIPANT=NB14_ROOT/"participant_app/participant_cases.json"
NB14_HTML=NB14_ROOT/"participant_app/index.html"
NB14_MODEL=NB14_ROOT/"data/distinct_case_model_results.json"
NB14_PREPARED=NB14_ROOT/"data/prepared_distinct_cases.json"

FINAL_ROOT=WISH_ROOT/"final_wish_pilot"
P_ROOT=FINAL_ROOT/"participant_app"
P_ASSETS=P_ROOT/"assets"
R_ROOT=FINAL_ROOT/"researcher_only"
D_ROOT=FINAL_ROOT/"docs"
E_ROOT=FINAL_ROOT/"evaluation"

for p in (P_ROOT,P_ASSETS,R_ROOT,D_ROOT,E_ROOT):
    p.mkdir(parents=True,exist_ok=True)

def now():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00","Z")

def loadj(p:Path)->Any:
    with p.open("r",encoding="utf-8") as f:
        return json.load(f)

def writej(p:Path,obj:Any):
    p.parent.mkdir(parents=True,exist_ok=True)
    with p.open("w",encoding="utf-8") as f:
        json.dump(obj,f,indent=2,ensure_ascii=False,allow_nan=False)
        f.write("\n")

required=[NB14_AUDIT,NB14_RESEARCH,NB14_PARTICIPANT,NB14_HTML,NB14_MODEL,NB14_PREPARED]
missing=[str(p) for p in required if not p.exists() or p.stat().st_size==0]
if missing:
    raise FileNotFoundError("Run Notebook 14 first. Missing:\n"+"\n".join(missing))

audit14=loadj(NB14_AUDIT)
if audit14.get("status")!="completed" or not audit14.get("final_gate"):
    raise RuntimeError("Notebook 14 did not pass its final gate.")

research14=loadj(NB14_RESEARCH)
participant14=loadj(NB14_PARTICIPANT)
model14=loadj(NB14_MODEL)
prepared14=loadj(NB14_PREPARED)

assert len(research14["cases"])==12
assert len(participant14["cases"])==12

print("✅ Notebook 14 gate passed")
print(f"✅ Final pilot workspace: {FINAL_ROOT}")


Mounted at /content/drive
✅ Notebook 14 gate passed
✅ Final pilot workspace: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot


In [2]:
# Cell 2 — Regenerate previews from MRI + MODEL prediction only

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

prepared={x["participant_case_id"]:x for x in prepared14["cases"]}
model={x["participant_case_id"]:x for x in model14["cases"]}

if P_ASSETS.exists():
    shutil.rmtree(P_ASSETS)
P_ASSETS.mkdir(parents=True,exist_ok=True)

preview_rows=[]

for case in participant14["cases"]:
    pid=case["case_id"]
    t1c_path=Path(prepared[pid]["modality_paths"]["T1c"])
    pred_path=Path(model[pid]["prediction_path"])

    if not t1c_path.exists() or not pred_path.exists():
        raise FileNotFoundError(f"{pid}: T1c or prediction missing")

    t1_img=nib.load(str(t1c_path))
    pred_img=nib.load(str(pred_path))
    t1=np.asanyarray(t1_img.dataobj).astype(np.float32,copy=False)
    pred=np.rint(np.asanyarray(pred_img.dataobj)).astype(np.uint8)

    assert t1.shape==pred.shape
    assert np.allclose(t1_img.affine,pred_img.affine,atol=1e-4)

    # Slice selection is based on the MODEL prediction, not expert reference.
    if np.any(pred>0):
        z=int(np.argmax((pred>0).sum(axis=(0,1))))
    else:
        z=t1.shape[2]//2

    fig=plt.figure(figsize=(5.2,5.2))
    plt.imshow(np.rot90(t1[:,:,z]),cmap="gray")
    if np.any(pred[:,:,z]>0):
        plt.contour(np.rot90(pred[:,:,z]>0),levels=[0.5],linewidths=1.2)
    plt.title("Current MRI + AI-derived segmentation evidence")
    plt.axis("off")
    plt.tight_layout()

    out_png=P_ASSETS/f"{pid}.png"
    fig.savefig(out_png,dpi=180,bbox_inches="tight")
    plt.close(fig)

    preview_rows.append({
        "case_id":pid,
        "preview_file":f"assets/{pid}.png",
        "slice_selection_basis":"model-predicted-whole-tumor-mask",
        "contour_basis":"model-predicted-whole-tumor-mask",
        "expert_reference_used":False,
    })

PREVIEW_AUDIT=E_ROOT/"final_preview_provenance.json"
writej(PREVIEW_AUDIT,{
    "generated_utc":now(),
    "preview_count":len(preview_rows),
    "previews":preview_rows,
    "expert_reference_used_in_participant_previews":False,
})

assert len(preview_rows)==12
assert all(not x["expert_reference_used"] for x in preview_rows)

print("✅ 12 previews regenerated")
print("✅ Expert-reference preview leakage: 0")


✅ 12 previews regenerated
✅ Expert-reference preview leakage: 0


In [3]:
# Cell 3 — Remove trajectory × presentation-condition confounding

research_cases=deepcopy(research14["cases"])
by_id={x["scenario_id"]:x for x in research_cases}

pairs=[
    ("S01","S02"),
    ("S03","S04"),
    ("S05","S06"),
    ("S07","S08"),
    ("S09","S10"),
    ("S11","S12"),
]

seqA={}
for pair_no,(a,b) in enumerate(pairs,start=1):
    pair=[by_id[a],by_id[b]]
    stable=next(x for x in pair if x["trajectory"]=="stable")
    progression=next(x for x in pair if x["trajectory"]=="progression")

    if pair_no%2==1:
        seqA[stable["scenario_id"]]="evidence-first"
        seqA[progression["scenario_id"]]="ai-first"
    else:
        seqA[stable["scenario_id"]]="ai-first"
        seqA[progression["scenario_id"]]="evidence-first"

for x in research_cases:
    a=seqA[x["scenario_id"]]
    x["condition_by_sequence"]={
        "A":a,
        "B":"ai-first" if a=="evidence-first" else "evidence-first",
    }

def condition_matrix(sequence):
    out={
        "evidence-first":{"stable":0,"progression":0},
        "ai-first":{"stable":0,"progression":0},
    }
    for x in research_cases:
        out[x["condition_by_sequence"][sequence]][x["trajectory"]]+=1
    return out

expected={
    "evidence-first":{"stable":3,"progression":3},
    "ai-first":{"stable":3,"progression":3},
}
assert condition_matrix("A")==expected
assert condition_matrix("B")==expected

updated={x["scenario_id"]:x for x in research_cases}
for seq in ("A","B"):
    for a,b in pairs:
        assert {
            updated[a]["condition_by_sequence"][seq],
            updated[b]["condition_by_sequence"][seq],
        }=={"evidence-first","ai-first"}

FINAL_RESEARCH=R_ROOT/"final_researcher_scenario_key.json"
writej(FINAL_RESEARCH,{
    "generated_utc":now(),
    "case_count":12,
    "cases":research_cases,
    "warning":"RESEARCHER ONLY — never distribute to participants.",
})

print("✅ Trajectory confounding removed")
print("✅ Sequence A:",condition_matrix("A"))
print("✅ Sequence B:",condition_matrix("B"))


✅ Trajectory confounding removed
✅ Sequence A: {'evidence-first': {'stable': 3, 'progression': 3}, 'ai-first': {'stable': 3, 'progression': 3}}
✅ Sequence B: {'evidence-first': {'stable': 3, 'progression': 3}, 'ai-first': {'stable': 3, 'progression': 3}}


In [4]:
# Cell 4 — Create final participant manifest and blocked allocation

participant_cases=deepcopy(participant14["cases"])
preview_map={x["case_id"]:x["preview_file"] for x in preview_rows}
for x in participant_cases:
    x["preview_file"]=preview_map[x["case_id"]]

FINAL_PARTICIPANT=P_ROOT/"participant_cases.json"
writej(FINAL_PARTICIPANT,{
    "generated_utc":now(),
    "project":"NeuroFHIR-Review",
    "case_count":12,
    "cases":participant_cases,
    "path_A_initial_labels":["Stable","Progression","Uncertain"],
    "path_B_initial_labels":["Evidence sufficient","Flag concern","Escalate for expert review"],
    "final_actions":["Accept AI","Keep initial judgment","Amend","Reject AI","Escalate"],
    "reason_codes":[
        "evidence-supports-ai",
        "ai-conflicts-with-longitudinal-evidence",
        "low-ai-confidence",
        "qc-failure",
        "missing-or-incomplete-provenance",
        "model-limitation-relevant",
        "evidence-incomplete",
        "uncertain-requires-expert-review",
        "other",
    ],
    "study_boundary":(
        "Formative research prototype. Public de-identified MRI plus standardized/"
        "synthetic longitudinal workflow context. No patient-care use."
    ),
    "participant_id_rule":(
        "Coordinator assigns sequential pseudonymous IDs P001, P002, ... "
        "Odd IDs map to hidden A; even IDs map to hidden B."
    ),
})

ALLOCATION=R_ROOT/"blocked_sequence_allocation.csv"
with ALLOCATION.open("w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=["participant_id","hidden_sequence"])
    w.writeheader()
    for i in range(1,41):
        w.writerow({
            "participant_id":f"P{i:03d}",
            "hidden_sequence":"A" if i%2==1 else "B",
        })

print("✅ Final participant manifest created")
print("✅ Blocked assignment table created: P001=A, P002=B, ...")


✅ Final participant manifest created
✅ Blocked assignment table created: P001=A, P002=B, ...


In [5]:
# Cell 5 — Patch the Notebook 14 app robustly for blocked allocation + balanced conditions

import re

source_html = NB14_HTML.read_text(encoding="utf-8")
patched = source_html


def replace_once(pattern: str, replacement: str, text: str, label: str) -> str:
    updated, count = re.subn(
        pattern,
        lambda match: replacement,
        text,
        count=1,
        flags=re.MULTILINE | re.DOTALL,
    )
    if count != 1:
        raise RuntimeError(
            f"Could not patch {label}: expected 1 match, found {count}. "
            "The Notebook 14 app source may have changed."
        )
    print(f"✅ Patched {label}")
    return updated


# A) Replace hash-based allocation with blocked odd/even allocation.
patched = replace_once(
    r'function assignSequence\(id\)\{return \(hashText\("sequence-"\+id\)%2===0\)\?"A":"B"\}',
    """function participantNumber(id){
 const m=id.match(/(\\d+)$/);
 return m?Number(m[1]):null
}
function assignSequence(id){
 const n=participantNumber(id);
 if(n===null||!Number.isInteger(n)||n<1)return null;
 return n%2===1?\"A\":\"B\"
}""",
    patched,
    "sequence allocation",
)

# B) Replace the old condition rule with the final balanced scenario map.
patched = replace_once(
    r'function conditionFor\(caseIndex,sequence\)\{\s*'
    r'const odd=\(caseIndex%2===0\);\s*'
    r'if\(sequence==="A"\) return odd\?"evidence-first":"ai-first";\s*'
    r'return odd\?"ai-first":"evidence-first";\s*'
    r'\}',
    """function conditionForScenario(scenarioId,sequence){
 const mapA={
  \"S01\":\"evidence-first\",\"S02\":\"ai-first\",
  \"S03\":\"ai-first\",\"S04\":\"evidence-first\",
  \"S05\":\"evidence-first\",\"S06\":\"ai-first\",
  \"S07\":\"ai-first\",\"S08\":\"evidence-first\",
  \"S09\":\"evidence-first\",\"S10\":\"ai-first\",
  \"S11\":\"ai-first\",\"S12\":\"evidence-first\"
 };
 const a=mapA[scenarioId];
 if(!a)throw new Error(\"Unknown scenario ID: \"+scenarioId);
 return sequence===\"A\"?a:(a===\"evidence-first\"?\"ai-first\":\"evidence-first\")
}""",
    patched,
    "scenario condition mapping",
)

# C) Make current-condition lookup use scenario ID instead of scenario index.
patched = replace_once(
    r'function condition\(\)\{return conditionFor\(originalScenarioIndex\(current\(\)\),S\.sequence\)\}',
    'function condition(){return conditionForScenario(current().scenario_id,S.sequence)}',
    patched,
    "current condition lookup",
)

# D) Enforce coordinator-issued sequential pseudonymous IDs at study start.
patched = replace_once(
    r'S\.participant=id;S\.tier=q\("#tier"\)\.value;S\.sequence=assignSequence\(id\);',
    """const seq=assignSequence(id);
 if(!seq){
  alert(\"Use the coordinator-assigned sequential pseudonymous ID, for example P001.\");
  return
 }
 S.participant=id;S.tier=q(\"#tier\").value;S.sequence=seq;""",
    patched,
    "blocked assignment at study start",
)

# E) Use the same final condition mapping in CSV export.
patched = replace_once(
    r'condition:conditionFor\(Number\(r\.scenario_id\.slice\(1\)\)-1,S\.sequence\),\.\.\.r',
    'condition:conditionForScenario(r.scenario_id,S.sequence),...r',
    patched,
    "CSV condition export",
)

# F) Update participant-ID instruction without exposing the hidden sequence.
patched = patched.replace(
    "Use the pseudonymous participant ID provided by the study coordinator. Do not enter PHI.",
    "Use only the sequential pseudonymous participant ID assigned by the study coordinator. Do not enter PHI.",
)

# G) Sanity checks.
required_new_snippets = [
    "function participantNumber(id)",
    'return n%2===1?"A":"B"',
    "function conditionForScenario(scenarioId,sequence)",
    '"S03":"ai-first","S04":"evidence-first"',
    "conditionForScenario(current().scenario_id,S.sequence)",
    "condition:conditionForScenario(r.scenario_id,S.sequence),...r",
]
missing_new = [snippet for snippet in required_new_snippets if snippet not in patched]
if missing_new:
    raise AssertionError("Final HTML is missing required patched logic: " + repr(missing_new))

obsolete_snippets = [
    'function assignSequence(id){return (hashText("sequence-"+id)%2===0)?"A":"B"}',
    "function conditionFor(caseIndex,sequence)",
    "conditionFor(originalScenarioIndex(current()),S.sequence)",
    "condition:conditionFor(Number(r.scenario_id.slice(1))-1,S.sequence),...r",
]
still_present = [snippet for snippet in obsolete_snippets if snippet in patched]
if still_present:
    raise AssertionError("Obsolete Notebook 14 logic still present: " + repr(still_present))

FINAL_HTML = P_ROOT / "index.html"
FINAL_HTML.write_text(patched, encoding="utf-8")

print("=" * 100)
print("✅ FINAL PARTICIPANT APP PATCH PASSED")
print("✅ Hash allocation removed")
print("✅ Hidden odd/even blocked allocation installed")
print("✅ Balanced scenario-specific condition mapping installed")
print(f"🌐 {FINAL_HTML}")
print("=" * 100)


✅ Patched sequence allocation
✅ Patched scenario condition mapping
✅ Patched current condition lookup
✅ Patched blocked assignment at study start
✅ Patched CSV condition export
✅ FINAL PARTICIPANT APP PATCH PASSED
✅ Hash allocation removed
✅ Hidden odd/even blocked allocation installed
✅ Balanced scenario-specific condition mapping installed
🌐 /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app/index.html


In [6]:
# Cell 6 — Final leakage, balance, and pilot-readiness audit

participant_text=FINAL_PARTICIPANT.read_text(encoding="utf-8").lower()
html_text=FINAL_HTML.read_text(encoding="utf-8")

forbidden_json=[
    "source_case_id",
    "case_type",
    "ai_correctness",
    "reference_workflow_disposition",
    "whole_tumor_dice",
    "source_reference_volume_ml",
    "path_a_reference_status",
    "wrong-but-plausible",
    "longitudinal-discordance",
]
json_leaks=[x for x in forbidden_json if x in participant_text]
assert not json_leaks,json_leaks

visible_terms=[
    ">Evidence-First<",
    ">AI-First<",
    "Counterbalance sequence",
    "wrong-but-plausible",
    "expected disposition",
]
visible_leaks=[x for x in visible_terms if x in html_text]
assert not visible_leaks,visible_leaks

bad_files=[
    str(p) for p in P_ROOT.rglob("*")
    if p.is_file() and (
        "reference" in p.name.lower()
        or str(p).lower().endswith(".nii")
        or str(p).lower().endswith(".nii.gz")
    )
]
assert not bad_files,bad_files

assert condition_matrix("A")==expected
assert condition_matrix("B")==expected

pilot_balance=[]
for n in range(5,16):
    a=(n+1)//2
    b=n//2
    assert abs(a-b)<=1
    pilot_balance.append({"n":n,"A":a,"B":b,"difference":abs(a-b)})

FINAL_AUDIT=E_ROOT/"notebook_14b_final_pilot_readiness_audit.json"
writej(FINAL_AUDIT,{
    "status":"completed",
    "audited_utc":now(),
    "notebook":"14B_NeuroFHIR_Review_WISH_Final_Stimulus_and_Counterbalance_Validation.ipynb",
    "metrics":{
        "distinct_cases":12,
        "expert_reference_preview_leakage":0,
        "participant_json_answer_leakage":len(json_leaks),
        "participant_visible_condition_leakage":len(visible_leaks),
        "participant_reference_or_nifti_files":len(bad_files),
        "sequence_A_trajectory_matrix":condition_matrix("A"),
        "sequence_B_trajectory_matrix":condition_matrix("B"),
        "blocked_assignment_n_5_to_15":pilot_balance,
    },
    "final_gate":True,
    "next_step":[
        "Internal full-session dry run",
        "Finalize reviewer training/rubric",
        "Complete applicable IRB/ethics/exemption process",
        "Recruit reviewers and collect real CSV/JSON exports",
        "Then run Notebook 15 human-AI analysis",
    ],
})

print("="*100)
print("✅ NOTEBOOK 14B FINAL GATE: TRUE")
print("✅ Expert-reference preview leakage: 0")
print("✅ Participant answer/performance leakage: 0")
print("✅ Visible condition/sequence leakage: 0")
print("✅ Evidence-First: 3 stable + 3 progression")
print("✅ AI-First:       3 stable + 3 progression")
print("✅ Blocked A/B allocation differs by at most 1 for N=5–15")
print(f"👤 Final pilot app: {FINAL_HTML}")
print(f"🔒 Researcher key: {FINAL_RESEARCH}")
print(f"📄 Audit: {FINAL_AUDIT}")
print("="*100)


✅ NOTEBOOK 14B FINAL GATE: TRUE
✅ Expert-reference preview leakage: 0
✅ Participant answer/performance leakage: 0
✅ Visible condition/sequence leakage: 0
✅ Evidence-First: 3 stable + 3 progression
✅ AI-First:       3 stable + 3 progression
✅ Blocked A/B allocation differs by at most 1 for N=5–15
👤 Final pilot app: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app/index.html
🔒 Researcher key: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/researcher_only/final_researcher_scenario_key.json
📄 Audit: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/evaluation/notebook_14b_final_pilot_readiness_audit.json


In [7]:
# Cell 7 — Coordinator handoff

README=D_ROOT/"FINAL_STUDY_COORDINATOR_README.md"
text=f"""# NeuroFHIR-Review — Final Pilot Coordinator Handoff

## Participant app
{P_ROOT / "index.html"}

Serve only the participant_app directory.

## Researcher-only material
{R_ROOT}

Never distribute that folder to participants.

## Participant IDs
Assign sequential pseudonymous IDs:
P001, P002, P003, ...

Odd-numbered IDs receive hidden Sequence A.
Even-numbered IDs receive hidden Sequence B.

Do not let participants choose their own ID.

## Final within-participant balance
Each participant receives:
- Evidence-First: 3 stable + 3 progression
- AI-First: 3 stable + 3 progression

## Imaging shown
Current public de-identified T1c MRI plus the MODEL-PREDICTED whole-tumor contour.
The expert reference mask is not used in the participant preview.

## Collect after each session
- neurofhir_review_<ID>.csv
- neurofhir_review_<ID>.json

## Before recruitment
Run one full internal dry run and complete the IRB/ethics/exemption process required by the institution conducting the study.

## After real pilot exports exist
Run:
15_NeuroFHIR_Review_WISH_Pilot_Data_Ingestion_and_Human_AI_Analysis.ipynb
"""
README.write_text(text,encoding="utf-8")
print(f"✅ Coordinator handoff: {README}")


✅ Coordinator handoff: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/docs/FINAL_STUDY_COORDINATOR_README.md
